一.多元线性模型的实现

多元线性回归则有多个特征：

y^=b0+b1x1+b2x2+⋯+bdxd

In [1]:
import numpy as np

# 学习时间
study_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8])

# 作业完成率
homework = np.array([50, 55, 60, 65, 70, 75, 80, 90])

# 出勤率
attendance = np.array([60, 65, 70, 72, 80, 85, 90, 95])

# 最终成绩
y = np.array([52, 55, 61, 65, 69, 74, 78, 82])

组合特征矩阵（8个样本，3个特征）

In [2]:
X = np.column_stack([
    study_hours,
    homework,
    attendance
])

print(X)
print("X.shape =", X.shape)

[[ 1 50 60]
 [ 2 55 65]
 [ 3 60 70]
 [ 4 65 72]
 [ 5 70 80]
 [ 6 75 85]
 [ 7 80 90]
 [ 8 90 95]]
X.shape = (8, 3)


训练多元线性回归模型（X.shape（8，1））

一元线性回归中X.reshape（-1，1）本意是转化为‘若干行（-1），1列的矩阵’，在上一个一元线性回归中即是把原来“一维的特征数组”，变成“每行一个样本、每列一个特征”的二维矩阵。即把[1,2,3,4,5,6,7,8]转为[[1],[2],[3],[4],[5],[6],[7],[8]]

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn import set_config

set_config(display="text")

model = LinearRegression()

model.fit(X, y)
print("截距 b0 =", model.intercept_)
print("系数 =", model.coef_)

截距 b0 = 55.10317460317462
系数 = [ 5.25396825 -0.10952381 -0.05555556]


In [ ]:
预测一个新样本

In [4]:
new_student = np.array([[6, 80, 90]])#表示一个学生、3个特征，所以两层括号

prediction = model.predict(new_student)

print("预测成绩 =", prediction[0])

预测成绩 = 72.86507936507937


二.固定两个特征，仅观察随一个特征变化，预测的变化（呈现线性关系，每增加1，增加量为b1），也就是说多元线性回归默认了特征可以分别看待，观察多重共线性（Multicollinearity，在多元线性回归中，多个特征之间存在很强的线性关系，以至于模型很难区分“到底是哪一个特征在起作用）。

多重共线性会导致回归系数变得不稳定->如果稍微改变训练集，回归系数可能有很大的变化，也就是说每个特征的贡献权重不稳定，多元线性回归无法准确区分哪个特征的贡献更大。

In [5]:
students = np.array([
    [4, 80, 90],
    [5, 80, 90],
    [6, 80, 90],
    [7, 80, 90],
    [8, 80, 90]
])

predictions = model.predict(students)

for x, pred in zip(students, predictions):
    print(
        "学习时间 =", x[0],
        "小时，预测成绩 =", pred
    )

学习时间 = 4 小时，预测成绩 = 62.35714285714285
学习时间 = 5 小时，预测成绩 = 67.61111111111111
学习时间 = 6 小时，预测成绩 = 72.86507936507937
学习时间 = 7 小时，预测成绩 = 78.11904761904762
学习时间 = 8 小时，预测成绩 = 83.37301587301587


计算三个特征之间的相关系数，三个系数强线性正相关

In [6]:
import pandas as pd

df = pd.DataFrame({
    "学习时间": study_hours,
    "作业完成率": homework,
    "出勤率": attendance
})

print(df.corr())

           学习时间     作业完成率       出勤率
学习时间   1.000000  0.994135  0.996348
作业完成率  0.994135  1.000000  0.991585
出勤率    0.996348  0.991585  1.000000


三.模型评价

In [9]:
y_pred = model.predict(X)

print("真实值：")
print(y)

print("\n预测值：")
print(y_pred)
errors = y - y_pred

print("误差：")
print(errors)

真实值：
[52 55 61 65 69 74 78 82]

预测值：
[51.54761905 55.97619048 60.4047619  65.         69.26190476 73.69047619
 78.11904762 82.        ]
误差：
[ 0.45238095 -0.97619048  0.5952381   0.         -0.26190476  0.30952381
 -0.11904762  0.        ]


计算MSE

In [15]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print("MSE =", mse)


MSE = 0.2113095238095237


R^2:决定系数:模型解释数据中“变化”的能力有多强。

In [16]:
print("R² =", r2)

R² = 0.9979078264969354


In [ ]:
随机划分训练集与测试集（random种子42）

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("训练集 X：", X_train.shape)
print("测试集 X：", X_test.shape)

训练集 X： (6, 3)
测试集 X： (2, 3)


In [18]:
model = LinearRegression()

model.fit(X_train, y_train)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
print("训练集 MSE：",
      mean_squared_error(y_train, y_train_pred))

print("测试集 MSE：",
      mean_squared_error(y_test, y_test_pred))

print("训练集 R²：",
      r2_score(y_train, y_train_pred))

print("测试集 R²：",
      r2_score(y_test, y_test_pred))

训练集 MSE： 0.0333333333333331
测试集 MSE： 1.0600000000000058
训练集 R²： 0.9996725784447477
测试集 R²： 0.9882548476454293


四.扩大实验数据集

In [20]:
import numpy as np

np.random.seed(42)

n = 200

# 两个特征
X1 = np.random.uniform(1, 10, n)
X2 = np.random.uniform(50, 100, n)

# 随机噪声
noise = np.random.normal(0, 3, n)

# 真实关系
y = 5 + 3 * X1 + 0.5 * X2 + noise#增加随机扰动

# 合并成特征矩阵
X = np.column_stack([X1, X2])

print("X.shape =", X.shape)
print("y.shape =", y.shape)
print("前5个样本：")
print(X[:5])

print("\n前5个真实标签：")
print(y[:5])

X.shape = (200, 2)
y.shape = (200,)
前5个样本：
[[ 4.37086107 82.10158231]
 [ 9.55642876 54.20699825]
 [ 7.58794548 58.0814357 ]
 [ 6.38792636 94.92770943]
 [ 2.40416776 80.32145298]]

前5个真实标签：
[63.07981078 60.83579692 58.8504132  70.69683352 53.34572884]


划分数据集（80%为训练集，20%为测试集）

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("训练集：", X_train.shape)
print("测试集：", X_test.shape)

训练模型

In [21]:
print("真实关系：")
print("y = 5 + 3*X1 + 0.5*X2")

print("\n模型学习到的参数：")
print("截距 =", model.intercept_)
print("X1系数 =", model.coef_[0])
print("X2系数 =", model.coef_[1])

真实关系：
y = 5 + 3*X1 + 0.5*X2

模型学习到的参数：
截距 = 49.599999999999966
X1系数 = 4.499999999999991
X2系数 = -0.03999999999999953


In [ ]:
用模型进行预测

In [22]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [23]:
from sklearn.metrics import mean_squared_error, r2_score

train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("训练集 MSE =", train_mse)
print("测试集 MSE =", test_mse)

print("训练集 R² =", train_r2)
print("测试集 R² =", test_r2)

训练集 MSE = 0.0333333333333331
测试集 MSE = 1.0600000000000058
训练集 R² = 0.9996725784447477
测试集 R² = 0.9882548476454293


五.多重共线性试验

In [24]:
import numpy as np

np.random.seed(42)

# 特征 X1
X1 = np.random.normal(0, 1, 100)

# 情况 A：X2 与 X1 基本独立
X2_independent = np.random.normal(0, 1, 100)

# 情况 B：X2 与 X1 高度相关
X2_correlated = X1 + np.random.normal(0, 0.1, 100)

print("独立情况下的相关系数：",
      np.corrcoef(X1, X2_independent)[0, 1])

print("高度相关情况下的相关系数：",
      np.corrcoef(X1, X2_correlated)[0, 1])

独立情况下的相关系数： -0.1364222121700025
高度相关情况下的相关系数： 0.993498846133746


In [25]:
noise = np.random.normal(0, 0.5, 100)

y = 3 * X1 + 2 * X2_independent + noise

In [27]:
from sklearn.linear_model import LinearRegression

# 模型 A：特征基本独立
X_independent = np.column_stack([X1, X2_independent])

model_A = LinearRegression()
model_A.fit(X_independent, y)


# 模型 B：特征高度相关
X_correlated = np.column_stack([X1, X2_correlated])

model_B = LinearRegression()
model_B.fit(X_correlated, y)
print("真实系数：")
print("X1 = 3")
print("X2 = 2")

print("\n独立特征模型：")
print(model_A.coef_)

print("\n高度相关特征模型：")
print(model_B.coef_)

真实系数：
X1 = 3
X2 = 2

独立特征模型：
[2.91437908 1.9807127 ]

高度相关特征模型：
[ 2.6890308  -0.05710194]


情况B的系数出现明显的偏离，是由于情况B中X1，X2高度相关，所以模型难以区分二者的独立贡献，导致了回归系数不稳定。

VIF（Variance Inflation Factor，方差膨胀因子），表示一个特征能在多大程度上被其他特征解释？
≈ 1	基本没有共线性
1～5	一般可以接受
> 5	值得关注
> 10	严重共线性

In [29]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 独立特征
vif_independent = [
    variance_inflation_factor(X_independent, i)
    for i in range(X_independent.shape[1])
]

# 高度相关特征
vif_correlated = [
    variance_inflation_factor(X_correlated, i)
    for i in range(X_correlated.shape[1])
]

print("独立特征 VIF：")
print(vif_independent)

print("\n高度相关特征 VIF：")
print(vif_correlated)

独立特征 VIF：
[np.float64(1.0194641453167685), np.float64(1.0194641453167683)]

高度相关特征 VIF：
[np.float64(77.47070794255468), np.float64(77.47070794255468)]


六.正则化（Ridge和Lasso）

In [30]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)#alpha表示正则强度

ridge.fit(X_correlated, y)

print("普通线性回归：")
print(model_B.coef_)

print("\nRidge：")
print(ridge.coef_)

普通线性回归：
[ 2.6890308  -0.05710194]

Ridge：
[1.76805112 0.82218981]


In [31]:
for alpha in [0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_correlated, y)
    
    print(
        "alpha =", alpha,
        "  coef =", ridge.coef_
    )

alpha = 0.01   coef = [ 2.6635639  -0.03252131]
alpha = 0.1   coef = [2.46972268 0.1543633 ]
alpha = 1   coef = [1.76805112 0.82218981]
alpha = 10   coef = [1.27795879 1.16950953]
alpha = 100   coef = [0.80733954 0.81594321]


Ridge会把系数向0压缩，让特征的重要性平均，本意是牺牲一点训练集上的拟合能力，换取模型更稳定、更简单、泛化能力更强。

Lasso（Ridge 是“把系数压小”，Lasso 是“把一部分系数压缩，甚至直接压到 0”）

In [32]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)

lasso.fit(X_correlated, y)

print("Lasso：")
print(lasso.coef_)

Lasso：
[2.3768241  0.12863608]
